# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [39]:
# Write your code below.
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get the PRICE_DATA variable
price_data_path = os.getenv("PRICE_DATA")

print("PRICE_DATA:", price_data_path)



PRICE_DATA: ../../05_src/data/prices/


In [40]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [41]:
import os
import glob

# Write your code below.
price_data_dir = os.getenv("PRICE_DATA")
parquet_files = glob.glob(os.path.join(price_data_dir, "**", "*.parquet"), recursive=True)


In [42]:
import pandas as pd

df = pd.read_parquet(parquet_files[0])
    
    # Display the first 5 rows
print(df.head())

Price        Date  Adj Close      Close       High        Low       Open  \
Ticker                                                                     
A      2000-01-03        NaN  43.382847  47.562963  40.596099  47.449986   
A      2000-01-04        NaN  40.068871  41.499902  39.014426  41.047995   
A      2000-01-05        NaN  37.583385  40.068861  36.340651  39.918225   
A      2000-01-06        NaN  36.152363  37.357444  35.022601  37.131491   
A      2000-01-07        NaN  39.165066  39.729947  35.549831  35.587487   

Price      Volume  Year  
Ticker                   
A       4674353.0  2000  
A       4765083.0  2000  
A       5758642.0  2000  
A       2534434.0  2000  
A       2819626.0  2000  


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [43]:
# Write your code below.

import dask.dataframe as dd

# Read all parquet files into a Dask DataFrame
ddf = dd.read_parquet(parquet_files)

# Reset index to make 'Ticker' a regular column
ddf = ddf.reset_index()

# Print available columns to verify 'Ticker' exists
print("Available columns:", ddf.columns)

# Ensure sorting by Ticker and Date for lag calculations
ddf = ddf.sort_values(by=["Ticker", "Date"])

# Add lag features for Close and Adj_Close
ddf["Close_lag_1"] = ddf.groupby("Ticker")["Close"].shift(1)
ddf["Adj_Close_lag_1"] = ddf.groupby("Ticker")["Adj Close"].shift(1)

# Calculate returns: (Close / Close_lag_1) - 1
ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1

# Calculate High-Low range
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

# Assign the result to dd_feat
dd_feat = ddf



Available columns: Index(['Ticker', 'Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'Year'],
      dtype='object', name='Price')


C:\Users\happy\AppData\Local\Temp\ipykernel_10104\610950266.py:18: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Close_lag_1"] = ddf.groupby("Ticker")["Close"].shift(1)
C:\Users\happy\AppData\Local\Temp\ipykernel_10104\610950266.py:19: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf["Adj_Close_lag_1"] = ddf.groupby("Ticker")["Adj Close"].shift(1)


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [52]:
dd_feat = dd_feat.groupby("Ticker").apply(lambda x: x.assign(ma_return = x["returns"].rolling(10).mean()))


ValueError: 'Ticker' is both an index level and a column label, which is ambiguous.

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Converting to Pandas is good since Pandas operations are straightforward and efficient for in-memory computations.
If the dataset is large, Converting to Pandas is not necessary and could even be problematic due to memory constraints.

Since Dask provides .rolling() and .mean() functions, we can compute the moving average return without converting to Pandas.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.